# DataSet

In [13]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as  plt
import os

class MIBCI2aDataset_withDA(Dataset):
    def _getFeatures(self, filePath):
        # implement the getFeatures method
        """
        read all the preprocessed data from the file path, read it using np.load,
        and concatenate them into a single numpy array
        """
        allFeatures = []
        for file in os.listdir(filePath):
            if file.endswith('.npy'):
                features = np.load(os.path.join(filePath, file))
                allFeatures.append(features)
        features = np.concatenate(allFeatures,axis=0)
        return  torch.tensor(features, dtype=torch.float32)
    def _getLabels(self, filePath):
        # implement the getLabels method
        """
        read all the preprocessed labels from the file path, read it using np.load,
        and concatenate them into a single numpy array
        """
        allLabels = []
        for file in os.listdir(filePath):
            if file.endswith('.npy'):
                labels = np.load(os.path.join(filePath, file))
                allLabels.append(labels)
        labels = np.concatenate(allLabels,axis=0)
        return torch.tensor(labels, dtype=torch.int64)
        

    def __init__(self, mode , method):
        # remember to change the file path according to different experiments
        assert mode in ['train', 'test', 'finetune']
        assert method in ['SD','LOSO']
        self.mode=mode
        if mode == 'train':
            # subject dependent: ./dataset/SD_train/features/ and ./dataset/SD_train/labels/
            # leave-one-subject-out: ./dataset/LOSO_train/features/ and ./dataset/LOSO_train/labels/
            if method == 'SD':
                self.features = self._getFeatures(filePath='./dataset/SD_train/features/')
                self.labels = self._getLabels(filePath='./dataset/SD_train/labels/')
            if method == 'LOSO':
                self.features = self._getFeatures(filePath='./dataset/LOSO_train/features/')
                self.labels = self._getLabels(filePath='./dataset/LOSO_train/labels/')
        if mode == 'finetune':
            # finetune: ./dataset/FT/features/ and ./dataset/FT/labels/
            self.features = self._getFeatures(filePath='./dataset/FT/features/')
            self.labels = self._getLabels(filePath='./dataset/FT/labels/')
        if mode == 'test':
            # subject dependent: ./dataset/SD_test/features/ and ./dataset/SD_test/labels/
            # leave-one-subject-out and finetune: ./dataset/LOSO_test/features/ and ./dataset/LOSO_test/labels/
            if method == 'SD':
                self.features = self._getFeatures(filePath='./dataset/SD_test/features/')
                self.labels = self._getLabels(filePath='./dataset/SD_test/labels/')
            if method == 'LOSO':
                self.features = self._getFeatures(filePath='./dataset/LOSO_test/features/')
                self.labels = self._getLabels(filePath='./dataset/LOSO_test/labels/')

    def __len__(self):
        return len(self.features)
        

    def __getitem__(self, idx):
        features =self.features[idx]
        labels = self.labels[idx]

        if self.mode == 'train' :
            features = self.normalize(features)
            
        return features ,labels
    
   
    
    def normalize(self, feature):
        
        mean = torch.mean(feature, dim=1, keepdims=True)
        std = torch.std(feature, dim=1, keepdims=True)
        feature = (feature - mean) / (std + 1e-7)  # Adding a small value to avoid division by zero   
        
        # add some noise
        if np.random.rand() > 0.5 :
            noise = torch.randn_like(feature) * 0.1  # 标准差为0.1的高斯噪声
            feature = feature + noise
    
        return feature

In [17]:
class MIBCI2aDataset(Dataset):
    def _getFeatures(self, filePath):
        # implement the getFeatures method
        """
        read all the preprocessed data from the file path, read it using np.load,
        and concatenate them into a single numpy array
        """
        allFeatures = []
        for file in os.listdir(filePath):
            if file.endswith('.npy'):
                features = np.load(os.path.join(filePath, file))
                allFeatures.append(features)
        features = np.concatenate(allFeatures,axis=0)
        return  torch.tensor(features, dtype=torch.float32)
    def _getLabels(self, filePath):
        # implement the getLabels method
        """
        read all the preprocessed labels from the file path, read it using np.load,
        and concatenate them into a single numpy array
        """
        allLabels = []
        for file in os.listdir(filePath):
            if file.endswith('.npy'):
                labels = np.load(os.path.join(filePath, file))
                allLabels.append(labels)
        labels = np.concatenate(allLabels,axis=0)
        return torch.tensor(labels, dtype=torch.int64)
        

    def __init__(self, mode , method):
        # remember to change the file path according to different experiments
        assert mode in ['train', 'test', 'finetune']
        assert method in ['SD','LOSO']
        self.mode=mode
        if mode == 'train':
            # subject dependent: ./dataset/SD_train/features/ and ./dataset/SD_train/labels/
            # leave-one-subject-out: ./dataset/LOSO_train/features/ and ./dataset/LOSO_train/labels/
            if method == 'SD':
                self.features = self._getFeatures(filePath='./dataset/SD_train/features/')
                self.labels = self._getLabels(filePath='./dataset/SD_train/labels/')
            if method == 'LOSO':
                self.features = self._getFeatures(filePath='./dataset/LOSO_train/features/')
                self.labels = self._getLabels(filePath='./dataset/LOSO_train/labels/')
        if mode == 'finetune':
            # finetune: ./dataset/FT/features/ and ./dataset/FT/labels/
            self.features = self._getFeatures(filePath='./dataset/FT/features/')
            self.labels = self._getLabels(filePath='./dataset/FT/labels/')
        if mode == 'test':
            # subject dependent: ./dataset/SD_test/features/ and ./dataset/SD_test/labels/
            # leave-one-subject-out and finetune: ./dataset/LOSO_test/features/ and ./dataset/LOSO_test/labels/
            if method == 'SD':
                self.features = self._getFeatures(filePath='./dataset/SD_test/features/')
                self.labels = self._getLabels(filePath='./dataset/SD_test/labels/')
            if method == 'LOSO':
                self.features = self._getFeatures(filePath='./dataset/LOSO_test/features/')
                self.labels = self._getLabels(filePath='./dataset/LOSO_test/labels/')

    def __len__(self):
        return len(self.features)
        

    def __getitem__(self, idx):
        features =self.features[idx]
        labels = self.labels[idx]

        if self.mode == 'train' :
            features = self.normalize(features)
            
        return features ,labels
    
   
    
    def normalize(self, feature):
        
        mean = torch.mean(feature, dim=1, keepdims=True)
        std = torch.std(feature, dim=1, keepdims=True)
        feature = (feature - mean) / (std + 1e-7)  # Adding a small value to avoid division by zero   
        
    
        return feature

In [ ]:
#大腦電極的運作
#(進行288次,電極通道數量,時間點) 9個人的資料
filePath = './dataset/LOSO_train/features/'
all_features = []
for file in os.listdir(filePath):
    if file.endswith('.npy'):
        features = np.load(os.path.join(filePath, file))
        all_features.append(features)
features = np.concatenate(all_features, axis=0)


filePath = './dataset/LOSO_train/labels/'
filepath = os.path.join(filePath, "s1E.npy")
labels = np.load(filepath)



print(features.shape)
print(labels.shape)
print(len(features))


# Model

In [15]:
class SquareLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x**2

class LogLayer(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return torch.log(x)

# 定義 SCCNet 類別
class SCCNet(nn.Module):
    def __init__(self, numClasses=4, timeSample=438, Nu=22, C=22, Nc=20, Nt=16, dropoutRate=0.5):
        super(SCCNet, self).__init__()
        
        # first layer
        self.conv1 = nn.Conv2d(1, Nu, (C, Nt), padding=0) #(batch size ,kernel ,height ,width)
        self.bn1 = nn.BatchNorm2d(Nu)
        
        
        # second layer
        self.conv2 = nn.Conv2d(1, Nc, (Nu, 12), padding=(0,6))
        self.bn2 = nn.BatchNorm2d(Nc)
        
        # square activation function
        self.square = SquareLayer()
        # dropout
        self.dropout = nn.Dropout(dropoutRate)
        self.logLayer =LogLayer()
        # pooling (use average pooling)
        self.pool = nn.AvgPool2d((1, 64), stride=(1, 12)) 
        
        # output shape (20,T/12)
        afterPoolSize = (timeSample-64)//12 
        self.fc = nn.Linear(Nc*afterPoolSize, numClasses) #620
        
     
    def forward(self, x):
        #print("Input shape:", x.shape)
        x = x.unsqueeze(1)  # (batch , 1 ,22,438)
        #print("After unsqueeze:", x.shape)
       
        ##############first layer####################
        x = self.conv1(x) #  (batch,22,1,438)
        #print("After conv1:", x.shape)
        x = self.bn1(x)
        #print("After bn1:", x.shape)
        x = self.dropout(x)
        x = x.permute(0,2, 1, 3) # [32, 1, 22, 438]
        #print("After permute:", x.shape)
        
        ##############Second layer####################
        x = self.conv2(x) #[32, 20, 1, 427]
        #print("After conv2:", x.shape)
        x = self.bn2(x) #[32, 20, 1, 427]
        #print("After bn2:", x.shape)
        x = self.square(x) # [32, 20, 1, 427]
        #print("After square:", x.shape)
        x = self.dropout(x) # [32, 20, 1, 427]
        #print("After dropout:", x.shape)
        
        x = x.permute(0,2, 1, 3)
        #print("After permute:", x.shape)
        ##############third layer####################
        x = self.pool(x) # [32, 20, 1, 31]
        #print("After pool:", x.shape)
        x=self.logLayer(x)
        #print("After logLayer:", x.shape)
        ##############forth layer##############
        x = x.flatten(1)   #[32, 620]
        
        #print("After flatten:", x.shape)
        x = self.fc(x) #[32, 4]
        #print("After fc:", x.shape)
       
        return x

# size

In [ ]:
train_set = MIBCI2aDataset(mode='train' ,method='LOSO')
test_set = MIBCI2aDataset(mode='test',method= 'LOSO')
print( len(train_set)) #2304 for SD  #4032 for LOSO
print( len(test_set)) # 2304 for SD #288 for LOSO

dataloader = DataLoader(train_set,batch_size=32,shuffle =True)

testloader = DataLoader(test_set,batch_size=32)


In [ ]:
import itertools

# 定义要调整的超参数范围
param_grid = {
    'lr': [0.001, 0.01, 0.1],           # 学习率的范围
    'batch_size': [288],          # 批量大小的范围
    'n_epochs': [100, 500, 1000]             # 训练周期数的范围
}
# Grid Search 函数
def grid_search(param_grid, model_class, model_args, train_func, test_func, mode, method):
    best_accuracy = 0
    best_params = {}
    
    for params in itertools.product(*param_grid.values()):
        params_dict = dict(zip(param_grid.keys(), params))
        lr = params_dict['lr']
        batch_size = params_dict['batch_size']
        n_epochs = params_dict['n_epochs']
        
        # 根据当前的超参数组合初始化模型
        model = model_class(numClasses=4,  # 假设有4个类别
                            timeSample=438,  # 时间采样点数
                            Nu=22,           # 第一个卷积层的输出通道数
                            C=22,            # 第一个卷积层的输入通道数
                            Nc=20,           # 第二个卷积层的输出通道数
                            Nt=16,           # 第一个卷积层的核大小在时间维度上的尺寸
                            dropoutRate=0.5) # Dropout 层的丢弃率
        model_path = f"model_lr{lr}_bs{batch_size}_ep{n_epochs}.pt"
        
        train(n_epochs=n_epochs, models=model, lr=lr, mode=mode, method=method, batch_size=batch_size, model_path=model_path)
        
        accuracy = test(mode=mode, method=method, batch_size=batch_size, model_path=model_path)
        
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_params = params_dict
    
    print("Best hyperparameters:", best_params)
    print("Best accuracy:", best_accuracy)

# 进行 Grid Search
grid_search(param_grid, 
            model_class=SCCNet,  # 使用你的模型类
            model_args=(),  # 空元组，因为 SCCNet 的构造函数不需要额外参数
            train_func=train,
            test_func=test,
            mode='train',  # 替换为你的模式
            method='SD')  # 替换为你的方法

# Train

In [3]:
from torch.optim.lr_scheduler import StepLR
def train_withDA(n_epochs ,models,lr,mode,method ,batch_size,model_path) :
    #################Prepare Data ###################
    
    #DataSet
    train_set = MIBCI2aDataset_withDA(mode=mode ,method=method)
    #DataLoader
    train_loader = DataLoader(train_set,batch_size=batch_size,shuffle =True)
    
    # "cuda" only when GPUs are available.
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Initialize a model, and put it on the device specified.
    model = models.to(device)

    #loss function
    criterion = nn.CrossEntropyLoss()

    # L2 regulation 
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4) #0.0001
    scheduler = StepLR(optimizer, step_size=50, gamma=0.1)
    
    
    # The number of training epochs.
    n_epochs = n_epochs

    softmax = nn.Softmax(dim=-1)

    train_losses = []

    for epoch in range(n_epochs):
        # These are used to record information in training.
        train_loss = []
        train_accs = []
        
        
        ###################training#############################
        model.train()
        for batch_idx, (features, labels) in enumerate(train_loader):
            features, labels = features.to(device), labels.to(device)
            
            # Gradients stored in the parameters in the previous step should be cleared out first.
            optimizer.zero_grad()
            # Forward the data. (Make sure data and model are on the same device.)
            
            logits = model(features)

            # Obtain the probability distributions by applying softmax on logits.
            probs = softmax(logits)
            
            # Calculate the cross-entropy loss.
            # We don't need to apply softmax before computing cross-entropy as it is done automatically.
            loss = criterion(logits, labels)
            

            # Compute the gradients for parameters.
            loss.backward()

            # Update the parameters with computed gradients.
            optimizer.step()

            # Compute the accuracy for current batch.
            acc = (logits.argmax(dim=-1) == labels).float().mean()

            # Record the loss and accuracy.
            train_loss.append(loss.item())
            train_accs.append(acc)
            
            
        # The average loss and accuracy of the training set is the average of the recorded values.
        train_loss = sum(train_loss) / len(train_loss)
        train_acc = sum(train_accs) / len(train_accs)
        
        train_losses.append(loss.item())
        
        # Print the information.
        
        print(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")
        #update lr
        #scheduler.step()
    
    #Save the model
    torch.save(model, model_path)
    print(f'model save to {model_path}')
    
    plot_learning(n_epochs,train_losses)
    
    return train_losses
    
    '''
    plt.figure()
    plt.plot(range(1, n_epochs + 1), train_losses, label='Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss Curve')
    plt.legend()  
    '''

In [4]:
def plot_learning(n_epochs, train_losses):
    plt.figure()
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss Curve')
    plt.plot(range(1, n_epochs + 1), train_losses, label='Training Loss')

In [75]:
def train(n_epochs ,models,lr,mode,method ,batch_size,model_path) :
    #################Prepare Data ###################
    
    #DataSet
    train_set = MIBCI2aDataset_withDA(mode=mode ,method=method)
    #DataLoader
    train_loader = DataLoader(train_set,batch_size=batch_size,shuffle =True)
    
    # "cuda" only when GPUs are available.
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Initialize a model, and put it on the device specified.
    model = models.to(device)

    #loss function
    criterion = nn.CrossEntropyLoss()

    # L2 regulation 
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4) #0.0001
    scheduler = StepLR(optimizer, step_size=50, gamma=0.1)
    
    
    # The number of training epochs.
    n_epochs = n_epochs

    softmax = nn.Softmax(dim=-1)

    train_losses = []

    for epoch in range(n_epochs):
        # These are used to record information in training.
        train_loss = []
        train_accs = []
        
        
        ###################training#############################
        model.train()
        for batch_idx, (features, labels) in enumerate(train_loader):
            features, labels = features.to(device), labels.to(device)
            
            # Gradients stored in the parameters in the previous step should be cleared out first.
            optimizer.zero_grad()
            # Forward the data. (Make sure data and model are on the same device.)
            
            logits = model(features)

            # Obtain the probability distributions by applying softmax on logits.
            probs = softmax(logits)
            
            # Calculate the cross-entropy loss.
            # We don't need to apply softmax before computing cross-entropy as it is done automatically.
            loss = criterion(logits, labels)
            

            # Compute the gradients for parameters.
            loss.backward()

            # Update the parameters with computed gradients.
            optimizer.step()

            # Compute the accuracy for current batch.
            acc = (logits.argmax(dim=-1) == labels).float().mean()

            # Record the loss and accuracy.
            train_loss.append(loss.item())
            train_accs.append(acc)
            
            
        # The average loss and accuracy of the training set is the average of the recorded values.
        train_loss = sum(train_loss) / len(train_loss)
        train_acc = sum(train_accs) / len(train_accs)
        
        train_losses.append(loss.item())
        
        # Print the information.
        
        print(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")
        
        #update lr
        #scheduler.step()
    
    #Save the model
        
    torch.save(model, model_path)
    print(f'model save to {model_path}')
            
    #plot_learning(n_epochs,train_losses)
    
    
    

In [80]:
test(method='LOSO',mode='test',batch_size=288,model_path='model_weight/model1.pt')

The training method is LOSO
Average Test Loss: 1.1405
Accuracy: 80.21%


0.8020833333333334

In [78]:
model =torch.load('model_weight/model_new79.17.pt')
for i in range (1,2000):
    if(train(n_epochs=1 ,lr=0.001,models=model,mode='finetune' ,method='LOSO' ,batch_size=288,model_path = 'model_weight/model1.pt')>0.8):
        break

[ Train | 001/001 ] loss = 0.00042, acc = 1.00000
The training method is LOSO
Average Test Loss: 1.3324
Accuracy: 78.12%
[ Train | 001/001 ] loss = 0.00012, acc = 1.00000
The training method is LOSO
Average Test Loss: 1.4069
Accuracy: 73.26%
[ Train | 001/001 ] loss = 0.00012, acc = 1.00000
The training method is LOSO
Average Test Loss: 1.3804
Accuracy: 75.69%
[ Train | 001/001 ] loss = 0.00034, acc = 1.00000
The training method is LOSO
Average Test Loss: 1.6203
Accuracy: 69.10%
[ Train | 001/001 ] loss = 0.00016, acc = 1.00000
The training method is LOSO
Average Test Loss: 1.3464
Accuracy: 75.35%
[ Train | 001/001 ] loss = 0.00006, acc = 1.00000
The training method is LOSO
Average Test Loss: 1.4341
Accuracy: 75.35%
[ Train | 001/001 ] loss = 0.00017, acc = 1.00000
The training method is LOSO
Average Test Loss: 1.4262
Accuracy: 74.31%
[ Train | 001/001 ] loss = 0.00020, acc = 1.00000
The training method is LOSO
Average Test Loss: 1.3919
Accuracy: 75.69%
[ Train | 001/001 ] loss = 0.000

In [46]:
model =torch.load('model_weight/model_new78.82.pt')
temp=0
i=0
while(temp<0.79):
    train(n_epochs=100 ,lr=0.001,models=model,mode='finetune' ,method='LOSO' ,batch_size=288,model_path = f'model_weight/model{i}.pt')
    temp=test(method='LOSO',mode='test',batch_size=288,model_path=f'model_weight/model{i}.pt')
    i+=1

[ Train | 001/100 ] loss = 0.00010, acc = 1.00000
[ Train | 002/100 ] loss = 0.00036, acc = 1.00000
[ Train | 003/100 ] loss = 0.00058, acc = 1.00000
[ Train | 004/100 ] loss = 0.00013, acc = 1.00000
[ Train | 005/100 ] loss = 0.00048, acc = 1.00000
[ Train | 006/100 ] loss = 0.00052, acc = 1.00000
[ Train | 007/100 ] loss = 0.00048, acc = 1.00000
[ Train | 008/100 ] loss = 0.00063, acc = 1.00000
[ Train | 009/100 ] loss = 0.00056, acc = 1.00000
[ Train | 010/100 ] loss = 0.00028, acc = 1.00000
[ Train | 011/100 ] loss = 0.00036, acc = 1.00000
[ Train | 012/100 ] loss = 0.00100, acc = 1.00000
[ Train | 013/100 ] loss = 0.00049, acc = 1.00000
[ Train | 014/100 ] loss = 0.00077, acc = 1.00000
[ Train | 015/100 ] loss = 0.00067, acc = 1.00000
[ Train | 016/100 ] loss = 0.00028, acc = 1.00000
[ Train | 017/100 ] loss = 0.00042, acc = 1.00000
[ Train | 018/100 ] loss = 0.00021, acc = 1.00000
[ Train | 019/100 ] loss = 0.00121, acc = 1.00000
[ Train | 020/100 ] loss = 0.00019, acc = 1.00000


KeyboardInterrupt: 

In [59]:
model = SCCNet()
#model = torch.load('model_weight/model_LL_LOSO_54.pt')
#model = SCCNet()
loss1 =train_withDA(n_epochs=200 ,lr=0.001,models=model,mode='train' ,method='SD' ,batch_size=288,model_path = 'model_weight/model1.pt')
model = SCCNet()
loss2 = train(n_epochs=200 ,lr=0.001,models=model,mode='train' ,method='SD' ,batch_size=288,model_path = 'model_weight/model2.pt')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.plot(range(1,201), loss1, label='SD with DA',color='blue')
plt.plot(range(1,201), loss2, label='SD',color='red')


plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

[ Train | 001/200 ] loss = 1.39352, acc = 0.26606
[ Train | 002/200 ] loss = 1.35212, acc = 0.32856
[ Train | 003/200 ] loss = 1.31876, acc = 0.37370
[ Train | 004/200 ] loss = 1.28792, acc = 0.40321
[ Train | 005/200 ] loss = 1.25940, acc = 0.42795
[ Train | 006/200 ] loss = 1.22715, acc = 0.45877
[ Train | 007/200 ] loss = 1.18882, acc = 0.47917
[ Train | 008/200 ] loss = 1.16281, acc = 0.47917
[ Train | 009/200 ] loss = 1.13464, acc = 0.50868


KeyboardInterrupt: 

# Test

In [79]:
def test(mode,method,batch_size,model_path):
    # "cuda" only when GPUs are available.
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ############## testing set###########################
    #load the testing set
    test_set = MIBCI2aDataset(mode=mode,method=method)
    testloader = DataLoader(test_set,batch_size=batch_size ,shuffle = False)
    
    # loss function
    criterion = nn.CrossEntropyLoss()
    #load the moedel
    model = torch.load(model_path)
    #define softmax
    softmax = nn.Softmax(dim=-1)
    # turn the model into eval mode
    model.eval()
    
    test_loss = 0
    correct = 0
    predictions = []
    true_labels = []
    with torch.no_grad():
        for features, labels in testloader:
            # transfer to GPU
            features, labels = features.to(device), labels.to(device)
            
            # Forward pass
            logits = model(features.to(torch.float32))
            preds =softmax(logits)
            # Calculate loss
            loss = criterion(logits, labels.to(torch.int64))
            test_loss += loss.item()  # Accumulate the total test loss
            
            # Get predicted labels
            preds = logits.argmax(dim=1)
            predictions.extend(preds.cpu().numpy())  # Store predicted labels
            true_labels.extend(labels.cpu().numpy())  # Store true labels for comparison
            
            # Calculate accuracy
            correct += (preds == labels).sum().item()
            
            
    # Average test loss
    avg_test_loss = test_loss / len(testloader)

    # Accuracy calculation
    accuracy = correct / len(testloader.dataset)
    if(mode == 'finetune'):
        print("The training method is LOSO with finetune")
    else:
        print(f"The training method is {method}")
    print(f"Average Test Loss: {avg_test_loss:.4f}")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    return accuracy
    

In [ ]:
test(method='SD',mode='test',batch_size=288,model_path='model_weight/model.pt')